# 06. TRANSFER LEARNING

What is transfer learning?

involves taking parameters of what one model has learned from another dataset and applying to our own problem
- Pretrained models = foundation models

In [1]:
import torch
from pathlib import Path
import torchinfo
import torchvision

from torchvision import transforms

print(torch.__version__)
print(torchvision.__version__)

2.14.0+cu132
0.29.0+cu132


Next, import the code from previous section

In [2]:
# setup device agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [3]:
!nvidia-smi

Fri Oct 30 13:33:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.71.05              Driver Version: 595.71.05      CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3050 ...    Off |   00000000:01:00.0 Off |                  N/A |
| N/A   52C    P3             15W /   30W |      14MiB /   4096MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Import the pizza steak sushi data that already have

In [4]:
train_dir = Path("../Datasets/pizza_steak_sushi/train")
test_dir = Path("../Datasets/pizza_steak_sushi/test")

train_dir, test_dir

(PosixPath('../Datasets/pizza_steak_sushi/train'),
 PosixPath('../Datasets/pizza_steak_sushi/test'))

## 2. create datasets and dataloaders

turn it into DataLoaders

To do so, could use `data_setup.py` and the `create_dataloaders()` func at going modular ipynb

there is one think need to think about when loading: how to **transform** it?

And with `torchvision` 0.13+ there's two ways to do this:
1. manual created transform - define what transforms you want your data to go through
2. automatically created transforms - the transforms for your data are defined by the model you'd like to use

Important point: when using a pretrained model, it's important that the data (including your custom data) that you pass through it is **transformed** in the same way that the data the model was trained on

In [5]:
import sys

# Add the repository root to Python's import path.
repo_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "going_modular").is_dir()),
    None,
)
if repo_root is None:
    raise FileNotFoundError(
        "Could not find the going_modular folder in the current directory or its parents."
    )

sys.path.insert(0, str(repo_root))
from going_modular import data_setup

# train_dataloader, test_dataloader, class_names = data_setup.create_dataloaders(
#     train_dir=train_dir,
#     test_dir=test_dir,
#     batch_size=32,
#     transform=None,
#     num_workers=2,
#     pin_memory=True,
# )

### 2.1 creating a transform for `torchvision.models` (manual creation)

`torchvision.models` containes pretrained models (models ready for transfer learning) right within `torchvision`

> All pre-trained models expect input images normalized in the same way, i.e. mini batches of 3-channel RGB images of shape (3 c H x W), where H and W are expected to be at least 224. The images have to be loaded in to a range of [0, 1] and then normalized using mean = [0.485, 0.456, 0.406] and std = [0.229, 0.224, 0.225]. You can use the following transform to normalize:
>
>

In [6]:
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

manual_transform = transforms.Compose([
    transforms.Resize((224, 224)), # resize image to 224, 224
    transforms.ToTensor(), # Turning image into a tensor
    normalize # make sure images have the same distribution as ImageNet (where our pretrained models were trained on)
])

In [7]:
train_dataloader, test_dataloader, class_names = data_setup.create_dataloaders(
    train_dir=train_dir,
    test_dir=test_dir,
    batch_size=32,
    transform=manual_transform,
)

train_dataloader, test_dataloader, class_names

(<torch.utils.data.dataloader.DataLoader at 0x74d5be929a90>,
 ['pizza', 'steak', 'sushi'])

### 2.2 creating a transform for `torchvision.models` (auto creation)

As of `torchvision` v0.13+ there is now support for automatic data transform creation based on the pretrained model 

In [8]:
# get a set of pretrained model weights
weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT # DEFAULT is the best weights for the model
weights

EfficientNet_B0_Weights.IMAGENET1K_V1

In [13]:
# get the transforms used to create our pretrained model weights
auto_transforms = weights.transforms()
auto_transforms

ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BICUBIC
)

In [10]:
# create dataloaders using automatic transforms
train_dataloader, test_dataloader, class_names = data_setup.create_dataloaders(
    train_dir=train_dir,
    test_dir=test_dir,
    batch_size=32,
    transform=auto_transforms,
)

train_dataloader, test_dataloader, class_names

(<torch.utils.data.dataloader.DataLoader at 0x74d5beaca990>,
 ['pizza', 'steak', 'sushi'])

## 3. Getting a pretrained model

There are various places to get a pretrained model, such as:
1. pytorch domain libraries
2. huggingface hub
3. paperswithcode (for models across different problem spaces/domains)
4. Libs like `timm` (torch image models)

### 3.1 Which pretrained model should you use?
*Experiment, experiment, experiment!*

The whole idea of transfer learning: take an already well performing model from a problem space similar to your own and then customize to your own problem

3 things to consider:
1. Speed - how fast does it run?
2. Size - how bog is the model?
3. Performance - how well does it go on your chosen problem

Where does the model live?
- Device (like a self driving car)
- On server?

Looking at https://docs.pytorch.org/vision/main/models.html 

Which model should we choose?

For our case (deploying FoodVision Mini on a mobile device), it looks like EffNetB0 is one of best options in terms of performance vs size

However, in light of the bitter lesson, if we had infinite compute, we'd likely pick the biggest model + more params + more general we could